In [12]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [14]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="llama-3.3-70b-versatile", model_provider="groq")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D992FAD120>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D992FAD030>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [19]:
from pydantic import Field, BaseModel

class Movie(BaseModel) :
    title :str = Field(description="Title of the movie.")
    year :int = Field(description="what year the movie was realeased?")
    director :str = Field(description="who directed the movie")
    actor : str = Field(description="who is the main lead actor of this movie?")
    actress : str = Field(description="Who is the main lead actress of this movie?")

In [20]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output.invoke("info about the inception movie")

Movie(title='Inception', year=2010, director='Christopher Nolan', actor='Leonardo DiCaprio', actress='Marion Cotillard')

#### raw message along with parsed message

In [24]:
from pydantic import Field, BaseModel

class Movie(BaseModel) :
    title :str = Field(..., description="Title of the movie.")
    year :int = Field(..., description="what year the movie was realeased?")
    director :str = Field(..., description="who directed the movie")
    actor : str = Field(..., description="who is the main lead actor of this movie?")
    actress : str = Field(..., description="Who is the main lead actress of this movie?")
    
model_with_raw_and_structured_output = model.with_structured_output(Movie, include_raw=True)
response = model_with_raw_and_structured_output.invoke("info about the inception movie")
print(response['raw'])
print(response['parsed'])

content='' additional_kwargs={'tool_calls': [{'id': '80j82cx0k', 'function': {'arguments': '{"actor":"Leonardo DiCaprio","actress":"Marion Cotillard","director":"Christopher Nolan","title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 304, 'total_tokens': 349, 'completion_time': 0.08140078, 'completion_tokens_details': None, 'prompt_time': 0.037987227, 'prompt_tokens_details': None, 'queue_time': 0.161637426, 'total_time': 0.119388007}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019fa348-a498-7801-9b7d-15e95294eec0-0' tool_calls=[{'name': 'Movie', 'args': {'actor': 'Leonardo DiCaprio', 'actress': 'Marion Cotillard', 'director': 'Christopher Nolan', 'title': 'Inception', 'year': 2010}, 'id': '80j82cx0k', 'type': 'tool_call'}] invalid_tool_c

### Nested structure

In [25]:
class Actor(BaseModel):
    name:str
    role:str

class Movie(BaseModel):
    title:str
    actors : list[Actor]
    genre: list[str]
    
nested_struct_model = model.with_structured_output(Movie)
response = nested_struct_model.invoke("info about the inception movie")
print(response)

title='Inception' actors=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')] genre=['Action', 'Sci-Fi']


### TypeDict

-TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [26]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title : Annotated[str, ..., "Title of the movie"]
    year : Annotated[int, ..., "year the movie was released"]
    director: Annotated[str, ..., "the director of the movie"]
    rating : Annotated[float, ..., "movies rating out of 10"]
    
model_with_structured_output_using_typeDict = model.with_structured_output(MovieDict)
response = model_with_structured_output_using_typeDict.invoke("info about the inception movie")
print(response)

{'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}


In [27]:
class Actor(TypedDict): # same as pydantic base model but without validation
    name:str
    role:str

class Movie(TypedDict):
    title:str
    actors : list[Actor]
    genre: list[str]
    
nested_struct_model = model.with_structured_output(Movie)
response = nested_struct_model.invoke("info about the avengers endgame movie")
print(response)

{'actors': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'}, {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'}], 'genre': ['Action', 'Adventure', 'Sci-Fi'], 'title': 'Avengers: Endgame'}


#### DataClasses
A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator.

In [33]:
from dataclasses import dataclass

@dataclass 
class ContactInfo:
    "contact information of the user"
    name:str # user name
    email:str # email of the user
    age:int # age of the user
    
model_with_data_class = model.with_structured_output(ContactInfo)
result = model_with_data_class.invoke("extract the information from : John Doe, john@gmail.com, 23")
print(result)

{'age': 23, 'email': 'john@gmail.com', 'name': 'John Doe'}
